# Generate the revision 4 YOLO dataset
Generate on the machine that will consume the data. Select one preset below; development is the main experiment. Generation refuses an existing destination. Asset preparation may download the existing generator's required assets. Run the experiment's Python environment as the notebook kernel.

In [ ]:
import json
import math
from itertools import combinations
from time import perf_counter

from tqdm import tqdm
from grid_detection.dataset.generator import GENERATOR_REVISION, generate_dataset
from grid_detection.dataset.yolo import YOLO_CLASSES
from grid_detection.research import DATA_DIR, show_yolo_examples

## Configuration
Choose `smoke` for a small pipeline check or `development` for training. Adjust workers for the host; all other values define this experiment.

In [ ]:
PRESET = "development"  # "smoke" or "development"
WORKERS = 8
configs = {
    "smoke": dict(name="smoke_v4", train=20, val=5, test=5),
    "development": dict(name="development_v1", train=2000, val=500, test=500),
}
config = configs[PRESET]
destination = DATA_DIR / config["name"]
if GENERATOR_REVISION != 4:
    raise RuntimeError(f"Expected generator revision 4, found {GENERATOR_REVISION}")
print(f"Selected {PRESET}: {config}; workers={WORKERS}; destination={destination}")

## Generate the selected dataset
The generator's default capture augmentation remains enabled. An existing path, including an incomplete run, is never overwritten. To retry, choose a new name or explicitly remove the old data yourself.

In [ ]:
if destination.exists():
    raise FileExistsError(f"Dataset destination already exists: {destination}")
start = perf_counter()
statistics = generate_dataset(
    destination, train=config["train"], val=config["val"], test=config["test"],
    seed=42, workers=WORKERS, png_compression=1, scene_details=True,
)
elapsed = perf_counter() - start
throughput = sum(config[s] for s in ("train", "val", "test")) / elapsed
print(f"Elapsed {elapsed:.1f} s; {throughput:.2f} images/s; {destination}")

## Generation summary
The statistics are also saved in `statistics.json`. Throughput above measures the full generation call, including asset preparation.

In [ ]:
for split, summary in statistics.items():
    counts = summary["counts"]
    print(f"{split}: {counts['scenes']} scenes, {counts['boards']} boards, "
          f"{counts['negatives']} negatives; {counts['pixels']:,} pixels; "
          f"{counts['encoded_bytes'] / 2**20:.1f} MiB PNG")
    print(f"  styles: {summary['board_styles']}; modes: {summary['scene_modes']}")
    print(f"  placement retries: {counts['placement_retries']}; "
          f"minefield overlap retries: {counts['minefield_overlap_retries']}")
print(f"Overall: {throughput:.2f} images/s")

## Integrity verification
Checks are streamed per manifest record and label file; board identity sets are retained only for split isolation. Empty labels are valid for negative screenshots.

In [ ]:
for name in ("data.yaml", "config.json", "manifest.jsonl", "statistics.json"):
    if not (destination / name).is_file():
        raise FileNotFoundError(destination / name)
metadata = json.loads((destination / "config.json").read_text())
expected = {"generator_revision": 4, "status": "complete", "seed": 42,
            "workers": WORKERS, "png_compression": 1, "scene_details": True,
            "augment": True, **{s: config[s] for s in ("train", "val", "test")}}
for key, value in expected.items():
    if metadata.get(key) != value:
        raise ValueError(f"{key}: expected {value!r}, found {metadata.get(key)!r}")
if metadata["classes"] != list(YOLO_CLASSES):
    raise ValueError("Unexpected class order")

splits = ("train", "val", "test")
image_stems, label_stems = {}, {}
for split in splits:
    image_stems[split] = {p.stem for p in (destination / "images" / split).glob("*.png")}
    label_stems[split] = {p.stem for p in (destination / "labels" / split).glob("*.txt")}
    if image_stems[split] != label_stems[split] or len(image_stems[split]) != config[split]:
        raise ValueError(f"Image/label stems or count mismatch in {split}")

identities = {split: set() for split in splits}
seen = {split: set() for split in splits}
tolerance = 1e-7  # labels are rounded to 8 decimal places
with (destination / "manifest.jsonl").open() as manifest:
    for line in tqdm(manifest, total=sum(config[s] for s in splits), desc="Verifying", unit="scene"):
        record = json.loads(line)
        split = record["split"]
        image = destination / record["image"]
        label = destination / record["label"]
        stem = image.stem
        if (split not in splits or image != destination / "images" / split / f"{stem}.png"
                or label != destination / "labels" / split / f"{stem}.txt"
                or stem not in image_stems[split] or stem in seen[split]):
            raise ValueError(f"Invalid or duplicate manifest entry: {record}")
        seen[split].add(stem)
        identities[split].update(record["board_ids"])
        lines = label.read_text().splitlines()
        if not record["board_ids"] and lines:
            raise ValueError(f"Negative scene has labels: {label}")
        for row in lines:
            fields = row.split()
            if len(fields) != 5 or fields[0] not in {str(i) for i in range(len(YOLO_CLASSES))}:
                raise ValueError(f"Invalid YOLO class or row: {label}: {row}")
            x, y, w, h = map(float, fields[1:])
            if (not all(math.isfinite(v) and 0 <= v <= 1 for v in (x, y, w, h))
                    or w <= 0 or h <= 0 or x-w/2 < -tolerance or y-h/2 < -tolerance
                    or x+w/2 > 1+tolerance or y+h/2 > 1+tolerance):
                raise ValueError(f"Invalid box: {label}: {row}")
for split in splits:
    if seen[split] != image_stems[split]:
        raise ValueError(f"Manifest does not cover {split}")
for a, b in combinations(splits, 2):
    if identities[a] & identities[b]:
        raise ValueError(f"Minefield identities overlap: {a} / {b}")
print("Counts, labels, manifest coverage and split isolation verified.")

## Visual inspection
Three evenly spaced annotated examples per split keep the output compact. Coverage of negatives, multi-game scenes and all styles is probabilistic; use the manifest to locate further examples as needed.

In [ ]:
for split in ("train", "val", "test"):
    show_yolo_examples(destination, split, YOLO_CLASSES, count=3)

# Select at most one scene per category, reusing a scene if it covers several.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

wanted = {"negative", "multiple games", "classic", "windows7", "microsoft", "google"}
selected = {}
with (destination / "manifest.jsonl").open() as manifest:
    for line in manifest:
        record = json.loads(line)
        categories = set()
        if not record["board_ids"]: categories.add("negative")
        if len(record["board_ids"]) > 1: categories.add("multiple games")
        for window in record["resolved_scene"]["windows"]:
            if window["type"] == "minesweeper":
                categories.add(window["content"]["appearance"]["style"])
        for category in wanted & categories:
            selected[category] = record
        wanted -= categories
        if not wanted: break
print(f"Unavailable categories: {sorted(wanted)}")
if selected:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, (category, record) in zip(axes.flat, selected.items()):
        with Image.open(destination / record["image"]) as image:
            ax.imshow(image)
            width, height = image.size
        for line in (destination / record["label"]).read_text().splitlines():
            class_id, x, y, w, h = map(float, line.split())
            class_id = int(class_id)
            left, top = (x-w/2)*width, (y-h/2)*height
            color = f"C{class_id}"
            ax.add_patch(Rectangle((left, top), w*width, h*height, fill=False, edgecolor=color))
            ax.text(left, top, YOLO_CLASSES[class_id], color="white", backgroundcolor=color, fontsize=7)
        ax.set_title(f"{category} — {record['split']}/{Path(record['image']).name}")
    fig.tight_layout()
    plt.show()